In [28]:
import ast
import numpy as np
import random
from scipy.optimize import fsolve
from datetime import datetime, timedelta

# Constants
MU = 1.32712440018e20  # Standard gravitational parameter for the Sun (m^3/s^2)
AU_TO_M = 1.496e11     # Conversion from AU to meters
START_DATE = datetime(2030, 1, 1)  # Set reference date to 1st January 2030
delta = 1e-6  # Tolerance for eccentric anomaly calculation

In [29]:
def spline(x,y,n,yp1,ypn):

    n=len(x)-1
    y2 = np.zeros(n+1)
    u = np.zeros(n+1)

    if (yp1>0.99e30):
        y2[1]=0.0
        u[1]=0.0
    else:
        y2[1]=-0.5
        u[1]=(3/(x[2]-x[1]))*((y[2]-y[1])/(x[2]-x[1])-yp1)

    for i in range(2,n):
        sig=(x[i]-x[i-1])/(x[i+1]-x[i-1])
        p=sig*y2[i-1]+2
        y2[i]=(sig-1)/p
        u[i]=(6*((y[i+1]-y[i])/(x[i+1]-x[i])-(y[i]-y[i-1])/(x[i]-x[i-1]))/(x[i+1]-x[i-1])-sig*u[i-1])/p

    if (ypn>0.99e30):
        qn=0.0
        un=0.0
    else:
        qn=0.5
        un=(3/(x[-1]-x[-2]))*(ypn-(y[-1]-y[-2])/(x[-1]-x[-2]))

    y2[-1]=(un-qn*u[-2])/(qn*y2[-2]+1)
    for k in range(n-1,0,-1):
        y2[k]=y2[k]*y2[k+1]+u[k]

    return y2


In [30]:
def splint(xa,ya,y2a,n,x):
    klo=1
    khi=n
    while (khi-klo>1):
        k=int((khi+klo)/2)
        if(xa[k]>x):
            khi=k
        else:
            klo=k

    h=xa[khi]-xa[klo]
    a=(xa[khi]-x)/h
    b=(x-xa[klo])/h
    return np.array(a*ya[klo]+b*ya[khi]+((a**3-a)*y2a[klo]+(b**3-b)*y2a[khi])*(h**2)/6)

In [31]:
"""
This routine calculates the MOID between two asteroids (or any objects). The orbital elements (in
AU and radians) are given in the arrays oe1 and oe2 as [a,e,i,Omega,omega].
It uses methods described in the paper by T. Wisniowski and H. Rickman,
"A Fast, Geometric Method for Calculating Accurate Minimum Orbit Intersection Distances (MOIDs),"
published in 2013 in Acta Astronomica.
The program is free and may be used without limits as the core of any other program.
The authors appreciate mentions if this program proves useful.
"""

def moid(oe1,oe2):
    # Initialize NumPy arrays equivalent to Fortran arrays
    rAt = np.zeros(4, dtype=np.float64)
    rBt = np.zeros(4, dtype=np.float64)
    Axt = np.zeros(4, dtype=np.float64)
    Ayt = np.zeros(4, dtype=np.float64)
    Bxt = np.zeros(4, dtype=np.float64)
    Byt = np.zeros(4, dtype=np.float64)
    Bzt = np.zeros(4, dtype=np.float64)

    tmpmoid = np.zeros(11, dtype=np.float64)
    tmptrueB = np.zeros(11, dtype=np.float64)
    tmplongit = np.zeros(11, dtype=np.float64)


    # Constants
    pi = np.pi
    twopi = 2.0 * pi
    deg = pi / 180.0

    # Parameters
    cstep = 0.12     # Scanning step of true anomaly/longitude in radians
    stepini = 0.07   # Initial step of first tuning in radians
    steptresh = 1e-5 # Final step of first tuning in radians
    stepmin = 1e-14  # Threshold step of second tuning in radians

    # Orbital parameters of asteroid A
    saxisA = oe1[0]   # Semi-major axis [AU]
    eccenA = oe1[1]   # Eccentricity
    argpeA = oe1[4]   # Argument of perihelion [rad]
    omegaA = oe1[3]   # Longitude of ascending node [rad]
    incliA = oe1[2]   # Inclination [rad]

    # Orbital parameters of asteroid B
    saxisB = oe2[0]   # Semi-major axis [AU]
    eccenB = oe2[1]   # Eccentricity
    argpeB = oe2[4]   # Argument of perihelion [rad]
    omegaB = oe2[3]   # Longitude of ascending node [rad]
    incliB = oe2[2]   # Inclination [rad]

    # Compute transition matrix components
    c11 = np.cos(omegaA) * np.cos(argpeA) - np.sin(omegaA) * np.cos(incliA) * np.sin(argpeA)
    c12 = np.sin(omegaA) * np.cos(argpeA) + np.cos(omegaA) * np.cos(incliA) * np.sin(argpeA)
    c13 = np.sin(incliA) * np.sin(argpeA)
    c21 = -np.cos(omegaA) * np.sin(argpeA) - np.sin(omegaA) * np.cos(incliA) * np.cos(argpeA)
    c22 = -np.sin(omegaA) * np.sin(argpeA) + np.cos(omegaA) * np.cos(incliA) * np.cos(argpeA)
    c23 = np.sin(incliA) * np.cos(argpeA)
    c31 = np.sin(incliA) * np.sin(omegaA)
    c32 = -np.sin(incliA) * np.cos(omegaA)
    c33 = np.cos(incliA)

    # Calculate new values of Euler angles using transition matrix
    sintmpi = np.sin(incliB)
    costmpi = np.cos(incliB)
    costmpo = np.cos(omegaB)
    sintmpo = np.sin(omegaB)
    costmpa = np.cos(argpeB)
    sintmpa = np.sin(argpeB)

    x1 = costmpo * costmpa - sintmpo * costmpi * sintmpa
    x2 = sintmpo * costmpa + costmpo * costmpi * sintmpa
    x3 = sintmpi * sintmpa
    y1 = -costmpo * sintmpa - sintmpo * costmpi * costmpa
    y2 = -sintmpo * sintmpa + costmpo * costmpi * costmpa
    y3 = sintmpi * costmpa
    z1 = sintmpi * sintmpo
    z2 = -sintmpi * costmpo
    z3 = costmpi

    z1n = c11 * z1 + c12 * z2 + c13 * z3
    z2n = c21 * z1 + c22 * z2 + c23 * z3
    z3n = c31 * z1 + c32 * z2 + c33 * z3
    y3n = c31 * y1 + c32 * y2 + c33 * y3
    x3n = c31 * x1 + c32 * x2 + c33 * x3

    incliB = np.arctan2(np.sqrt(z1n**2 + z2n**2), z3n)
    omegaB = -np.arctan2(z1n, -z2n)
    argpeB = -np.arctan2(x3n, y3n)

    # Precomputed values
    costmpo = np.cos(omegaB)
    sintmpo = np.sin(omegaB)
    sintmpi = np.sin(incliB)
    costmpi = z3n
    sint = sintmpo * costmpi
    cost = costmpo * costmpi

    radA = saxisA * (1.0 - eccenA**2)
    radB = saxisB * (1.0 - eccenB**2)

    # Initialize parameters
    trueB = -2.0 * cstep
    moid = 1e6  # Large initial MOID value
    dist_o = 1e6  # Large initial distance
    tmpmoid = [1e6] * 10  # Array for storing temporary MOID values
    iii1 = 0
    jjj1 = 0

    # Looking for the minima with rotating meridional plane
    # a) First, we calculate the coordinates of two additional positions of the plane to create the first triplet
    for iii in range(1, 3):
        rB = radB / (1.0 + eccenB * np.cos(trueB))  # Compute the radius for B
        sintmp = np.sin(trueB + argpeB)
        costmp = np.cos(trueB + argpeB)
        Bz_sq = (sintmpi * sintmp) ** 2  # Square of Z-coordinate for B

        longit = np.arctan2(sintmpo * costmp + sintmp * cost,
                             costmpo * costmp - sintmp * sint)  # Compute longitude for A
        tmp2 = eccenA * np.cos(longit)
        rA = radA / (1.0 + tmp2)  # Compute the radius for A (two possibilities)
        rA2 = radA / (1.0 - tmp2)
        tmp1 = rB * np.sqrt(1.0 - Bz_sq)

        # Choose the correct radius for A
        if abs(tmp1 - rA) > abs(tmp1 + rA2):
            rA = rA2
            longit -= np.pi  # The second possibility gives a smaller distance
            tmp1 += rA2
        else:
            tmp1 -= rA

        dist = rB**2 * Bz_sq + tmp1**2  # Square of the distance A-B

        if iii == 1:
            dist_oo = dist
        else:
            dist_o = dist
            trueB_o = trueB
            longit_o = longit

        trueB += cstep

    # b) Now, we scan one full revolution of the meridional plane
    nmax = 0  # Counts the minima
    dist_min = dist

    while trueB < (twopi + cstep):  # Loop for true anomaly of B
        rB = radB / (1.0 + eccenB * np.cos(trueB))  # Compute the radius for B
        sintmp = np.sin(trueB + argpeB)
        costmp = np.cos(trueB + argpeB)
        Bz_sq = (sintmpi * sintmp) ** 2  # Square of Z-coordinate for B

        longit = np.arctan2(sintmpo * costmp + sintmp * cost,
                             costmpo * costmp - sintmp * sint)  # Compute longitude for A
        tmp2 = eccenA * np.cos(longit)
        rA = radA / (1.0 + tmp2)  # Compute the radius for A (two possibilities)
        rA2 = radA / (1.0 - tmp2)
        tmp1 = rB * np.sqrt(1.0 - Bz_sq)

        # Choose the correct radius for A
        if abs(tmp1 - rA) > abs(tmp1 + rA2):
            rA = rA2
            longit -= np.pi  # The second possibility gives a smaller distance
            tmp1 += rA2
        else:
            tmp1 -= rA

        dist = rB**2 * Bz_sq + tmp1**2  # Square of the distance A-B

        # Check if a minimum was found
        if (dist_o <= dist) and (dist_o <= dist_oo):
            nmax += 1
            tmptrueB[nmax] = trueB_o
            tmplongit[nmax] = longit_o
            tmpmoid[nmax] = dist_o

        if dist_min > dist:
            dist_min = dist

        dist_oo = dist_o
        trueB_o = trueB
        longit_o = longit
        dist_o = dist
        trueB += cstep  # Increment true anomaly

    # End of scanning phase

    # "WATER" PROCEDURE
    # If only one minimum was detected, introduce additional points to avoid missing the global minimum

    if nmax < 2:  # Only one minimum was detected
        nmax = 4  # Use four evenly distributed points

        for iii in range(1, 5):  # Loop from 1 to 4 (inclusive)
            tmptrueB[iii] = (0.25 + 0.5 * iii) * np.pi  # Evenly distributed points

            sintmp = np.sin(tmptrueB[iii] + argpeB)
            costmp = np.cos(tmptrueB[iii] + argpeB)

            # Compute the longitude for A
            tmplongit[iii] = np.arctan2(sintmpo * costmp + sintmp * cost,
                                         costmpo * costmp - sintmp * sint)

            tmpmoid[iii] = 1e6  # Large initial MOID value


    # PARALLEL TUNING PROCEDURE
    # After the scanning phase, we refine the minima to find the absolute MOID.

    for jjj in range(1, nmax + 2):  # Loop from 1 to nmax+1
        if jjj <= nmax:
            moid = tmpmoid[jjj]
            trueB_m = tmptrueB[jjj]
            longit_m = tmplongit[jjj]
            step = stepini
            threshold = steptresh
        else:
            if nmax == 2:
                if abs(tmpmoid[1] - tmpmoid[2]) < 1e-4:
                    nmax = 1
                    # Go back to the water procedure
                    # Simulating Fortran's `goto 405` by calling the function again
                    #water_procedure()
                else:
                    if tmpmoid[1] < moid:
                        moid = tmpmoid[1]
                        trueB_m = tmptrueB[1]
                        longit_m = tmplongit[1]
            else:
                for iii in range(1, nmax):  # Choosing the best moid for final tuning
                    if tmpmoid[iii] < moid:
                        moid = tmpmoid[iii]
                        trueB_m = tmptrueB[iii]
                        longit_m = tmplongit[iii]
            step = 2.0 * stepini  # Initial step for final tuning
            threshold = stepmin   # Terminal step for final tuning

        rBt[2] = radB / (1.0 + eccenB * np.cos(trueB_m))
        sintmp = np.sin(trueB_m + argpeB)
        costmp = np.cos(trueB_m + argpeB)
        Bxt[2] = costmpo * costmp - sintmp * sint
        Byt[2] = sintmpo * costmp + sintmp * cost
        Bzt[2] = sintmpi * sintmp

        rAt[2] = radA / (1.0 + eccenA * np.cos(longit_m))
        Axt[2] = np.cos(longit_m)
        Ayt[2] = np.sin(longit_m)

        aleft = aright = bleft = bright = True

        while step >= threshold:
            lpoints = 0
            j1min, j1max = 1, 3
            i1min, i1max = 1, 3
            calc1 = calc2 = calc3 = calc4 = False

            if bleft:
                rBt[1] = radB / (1.0 + eccenB * np.cos(trueB_m - step))
                sintmp = np.sin(trueB_m - step + argpeB)
                costmp = np.cos(trueB_m - step + argpeB)
                Bxt[1] = costmpo * costmp - sintmp * sint
                Byt[1] = sintmpo * costmp + sintmp * cost
                Bzt[1] = sintmpi * sintmp
                lpoints += 1

            if bright:
                rBt[3] = radB / (1.0 + eccenB * np.cos(trueB_m + step))
                sintmp = np.sin(trueB_m + step + argpeB)
                costmp = np.cos(trueB_m + step + argpeB)
                Bxt[3] = costmpo * costmp - sintmp * sint
                Byt[3] = sintmpo * costmp + sintmp * cost
                Bzt[3] = sintmpi * sintmp
                lpoints += 1

            if aleft:
                rAt[1] = radA / (1.0 + eccenA * np.cos(longit_m - step))
                Axt[1] = np.cos(longit_m - step)
                Ayt[1] = np.sin(longit_m - step)
                lpoints += 1

            if aright:
                rAt[3] = radA / (1.0 + eccenA * np.cos(longit_m + step))
                Axt[3] = np.cos(longit_m + step)
                Ayt[3] = np.sin(longit_m + step)
                lpoints += 1

            j1_t, i1_t = 2, 2

            if lpoints == 1:
                if aleft:
                    i1max = 1
                if aright:
                    i1min = 3
                if bleft:
                    j1max = 1
                if bright:
                    j1min = 3

            if lpoints == 2:
                calc1 = aleft and bright
                calc2 = aleft and bleft
                calc3 = aright and bright
                calc4 = aright and bleft

            for j1 in range(j1min, j1max + 1):
                for i1 in range(i1min, i1max + 1):
                    if lpoints == 2:
                        if i1 != 1 and ((j1 != 3 and calc1) or (j1 != 1 and calc2)):
                            continue
                        if i1 != 3 and ((j1 != 3 and calc3) or (j1 != 1 and calc4)):
                            continue
                    if i1 == 2 and j1 == 2:
                        continue

                    Dx = rBt[j1] * Bxt[j1] - rAt[i1] * Axt[i1]
                    Dy = rBt[j1] * Byt[j1] - rAt[i1] * Ayt[i1]
                    Dz = rBt[j1] * Bzt[j1]
                    dist = Dx**2 + Dy**2 + Dz**2

                    if dist < moid:
                        moid = dist
                        j1_t, i1_t = j1, i1

            if j1_t != 2 or i1_t != 2:
                aleft = aright = bleft = bright = False

                if i1_t != 2:
                    if i1_t == 1:
                        aleft = True
                        longit_m -= step
                        rAt[3], Axt[3], Ayt[3] = rAt[2], Axt[2], Ayt[2]
                        rAt[2], Axt[2], Ayt[2] = rAt[1], Axt[1], Ayt[1]
                    else:
                        aright = True
                        longit_m += step
                        rAt[1], Axt[1], Ayt[1] = rAt[2], Axt[2], Ayt[2]
                        rAt[2], Axt[2], Ayt[2] = rAt[3], Axt[3], Ayt[3]

                if j1_t != 2:
                    if j1_t == 1:
                        bleft = True
                        trueB_m -= step
                        rBt[3], Bxt[3], Byt[3], Bzt[3] = rBt[2], Bxt[2], Byt[2], Bzt[2]
                        rBt[2], Bxt[2], Byt[2], Bzt[2] = rBt[1], Bxt[1], Byt[1], Bzt[1]
                    else:
                        bright = True
                        trueB_m += step
                        rBt[1], Bxt[1], Byt[1], Bzt[1] = rBt[2], Bxt[2], Byt[2], Bzt[2]
                        rBt[2], Bxt[2], Byt[2], Bzt[2] = rBt[3], Bxt[3], Byt[3], Bzt[3]

            else:
                aleft = aright = bleft = bright = True
                step *= 0.15  # Optimal reduction factor

        if jjj <= nmax:
            tmpmoid[jjj] = moid
            tmptrueB[jjj] = trueB_m
            tmplongit[jjj] = longit_m

    # Final MOID computation
    moid = np.sqrt(moid)
    return moid

In [32]:
def solve_kepler(mean_anomaly, eccentricity, tol=1e-6, max_iter=1000):
    # Initial guess: Use mean anomaly as first approximation
    E = mean_anomaly
    for _ in range(max_iter):
        f_E = E - eccentricity * np.sin(E) - mean_anomaly
        f_E_prime = 1 - eccentricity * np.cos(E)
        E_new = E - f_E / f_E_prime  # Newton-Raphson update

        if abs(E_new - E) < tol:
            return E_new
        E = E_new
    raise RuntimeError("Newton-Raphson method did not converge")

def true_anomaly(mean_anomaly, semi_major_axis, eccentricity):
    # Solve for eccentric anomaly using Newton-Raphson method
    E = solve_kepler(mean_anomaly, eccentricity)

    # Compute the true anomaly from eccentric anomaly
    true_anom = 2 * np.arctan2(
        np.sqrt(1 + eccentricity) * np.sin(E / 2),
        np.sqrt(1 - eccentricity) * np.cos(E / 2)
    )
    return true_anom

def orbital_period(a):
    a_m = a # Convert AU to meters
    P = 2 * np.pi * np.sqrt(a_m**3 / MU)
    return P

def eccentric_anomaly(e, M, delta):
    E = M
    while True:
        E_new = E + (M - E + e * np.sin(E)) / (1 - e * np.cos(E))
        if abs(E_new - E) < delta:
            break
        E = E_new
    return E

def static_elements(a,e,i,Omega,omega,theta0,mu):
    P = orbital_period(a)
    lvec = np.sqrt(a*(1-e**2)*mu)
    E0 = np.arctan2(np.sqrt(1-e**2)*np.sin(theta0),e+np.cos(theta0))
    M0 = E0-e*np.sin(E0)
    t0 = P*M0/(2*np.pi)
    R1 = np.array([[np.cos(Omega), -np.sin(Omega), 0],
                   [np.sin(Omega), np.cos(Omega), 0],
                   [0, 0, 1]])
    R2 = np.array([[1, 0, 0],
                   [0, np.cos(i), -np.sin(i)],
                   [0, np.sin(i), np.cos(i)]])
    R3 = np.array([[np.cos(omega), -np.sin(omega), 0],
                   [np.sin(omega), np.cos(omega), 0],
                   [0, 0, 1]])
    R = R1 @ R2 @ R3
    return P, lvec, t0, R

def time_evolution(a1,e1,a2,e2,P1,P2,t01,t02,R1,R2,lvec1,lvec2,dt,delta,mu,n,diameter,file):
    count = 0
    app = 0
    timesin = []
    timemin = []
    timemin_n = []
    timesout = []
    index = []
    mindist = 1e15
    for j, t_step in enumerate(dt):
        t1 = t01 + t_step
        t2 = t02 + t_step
        M1 = 2 * np.pi * t1 / P1
        M2 = 2 * np.pi * t2 / P2
        E1 = eccentric_anomaly(e1, M1, delta)
        E2 = eccentric_anomaly(e2, M2, delta)
        theta1 = np.arctan2(np.sqrt(1-e1**2)*np.sin(E1),np.cos(E1)-e1)
        theta2 = np.arctan2(np.sqrt(1-e2**2)*np.sin(E2),np.cos(E2)-e2)
        rval1 = a1*(1-e1**2)/(1+e1*np.cos(theta1))
        rval2 = a2*(1-e2**2)/(1+e2*np.cos(theta2))
        vval1 = mu/lvec1
        vval2 = mu/lvec2
        r_vec1 = [rval1*np.cos(theta1),rval1*np.sin(theta1),0]
        r_vec2 = [rval2*np.cos(theta2),rval2*np.sin(theta2),0]
        v_vec1 = [vval1*(-np.sin(theta1)),vval1*(e1+np.cos(theta1)),0]
        v_vec2 = [vval2*(-np.sin(theta2)),vval2*(e2+np.cos(theta2)),0]
        x1 = R1[0][0]*r_vec1[0]+R1[0][1]*r_vec1[1]+R1[0][2]*r_vec1[2]
        y1 = R1[1][0]*r_vec1[0]+R1[1][1]*r_vec1[1]+R1[1][2]*r_vec1[2]
        z1 = R1[2][0]*r_vec1[0]+R1[2][1]*r_vec1[1]+R1[2][2]*r_vec1[2]
        x2 = R2[0][0]*r_vec2[0]+R2[0][1]*r_vec2[1]+R2[0][2]*r_vec2[2]
        y2 = R2[1][0]*r_vec2[0]+R2[1][1]*r_vec2[1]+R2[1][2]*r_vec2[2]
        z2 = R2[2][0]*r_vec2[0]+R2[2][1]*r_vec2[1]+R2[2][2]*r_vec2[2]
        vx1 = R1[0][0]*v_vec1[0]+R1[0][1]*v_vec1[1]+R1[0][2]*v_vec1[2]
        vy1 = R1[1][0]*v_vec1[0]+R1[1][1]*v_vec1[1]+R1[1][2]*v_vec1[2]
        vz1 = R1[2][0]*v_vec1[0]+R1[2][1]*v_vec1[1]+R1[2][2]*v_vec1[2]
        vx2 = R2[0][0]*v_vec2[0]+R2[0][1]*v_vec2[1]+R2[0][2]*v_vec2[2]
        vy2 = R2[1][0]*v_vec2[0]+R2[1][1]*v_vec2[1]+R2[1][2]*v_vec2[2]
        vz2 = R2[2][0]*v_vec2[0]+R2[2][1]*v_vec2[1]+R2[2][2]*v_vec2[2]
        dist = np.sqrt((x1-x2)**2+(y1-y2)**2+(z1-z2)**2)
        vrel = np.sqrt((vx1-vx2)**2+(vy1-vy2)**2+(vz1-vz2)**2)

        if dist / 400000000 < 5 and count == 0:  # Entering 1LD
            count += 1
            app += 1
            timesin.append(dt[j])
            mindist = dist
            minvrel = vrel
            entry_time = convert_time_to_date(START_DATE, dt[j])
            closest_time = entry_time
            file.write("{0} {1} {2} {3} {4:.10f} ".format(n + 1, entry_time.strftime('%d'), entry_time.strftime('%m'), entry_time.strftime('%Y'), dist / 400000000))

        if dist / 400000000 < 5 and count == 1:  # Still inside 1LD
            if dist < mindist:
                mindist = dist
                minvrel = vrel
                closest_time = convert_time_to_date(START_DATE, dt[j])

        if dist / 400000000 > 5 and count == 1:  # Exiting 1LD
            count += -1
            timesout.append(dt[j])
            timemin.append(closest_time)
            exit_time = convert_time_to_date(START_DATE, dt[j])
            file.write("{0} {1} {2} {3:.10f} {4:.10f} {5:.2f} ".format(closest_time.strftime('%d'), closest_time.strftime('%m'), closest_time.strftime('%Y'), mindist / 400000000, minvrel / 1000, 1000 * diameter[n])) #
            file.write("{0} {1} {2} {3:.10f}\n".format(exit_time.strftime('%d'), exit_time.strftime('%m'), exit_time.strftime('%Y'), dist / 400000000))

    return timesin, timemin, timesout, mindist

def convert_time_to_date(start_date, time_seconds):
    return start_date + timedelta(seconds=time_seconds)

In [33]:
# translated from the fortran code by David Nesvorný
#https://arxiv.org/html/2404.18805v1
#https://www.boulder.swri.edu/~davidn/NEOMOD_Simulator

mh=52+1
ma=42+1
me=25+1
mi=22+1
ms=12+1

model=np.zeros((mh,ma,me,mi))
alpha=np.zeros((mh,ma,me,mi,ms))
nlogp=np.zeros(101)
y2=np.zeros(101)

# Read input parameters from neomod.par
filename="input_neomod3.dat"

# absolute magnitude binning: # of H bins, Hmin,Hmax
nh=52
minh=15.0
maxh=28.0
dh=(maxh-minh)/nh


# reference diameter Dref, # of NEOs with D>D_ref (from the MultiNest fit)
dref=1.0
nref=871.05
ldref=np.log10(dref)

# size distribution - # of segments
nseg=8
npoint=nseg+1

# segment boundaries in log10(D)
dpoint=np.array([0,-3.00000,-2.18396,-1.58396,-1.38396,-0.58396,-0.08396,0.11603,1.01603,2.00000])
mind2=10.e0**dpoint[1]
maxd2=10.e0**dpoint[npoint]

# segment slopes
gamma=np.array([0,-2.52421, -2.52421, -2.77526, -1.50274, -1.73718, -1.76720, -2.59529, -2.59529])

na=42 ; mina=0.0 ; maxa=4.2   # semimajor axis binning: # of a bins, amin (au), amax(au)
ne=25 ; mine=0.0 ; maxe=1.0   # eccentricity binning: # of e bins, emin, emax
ni=22 ; mini=0.0 ; maxi=88.0  # orbital inclination binning: # of i bins, imin (deg), imax (deg)
nsour=12           # number of model sources


#Read the model file
with open(filename,"r") as f:
    Lines = f.readlines()

Lines=Lines[13:]

for ll in Lines:

    rl =[ast.literal_eval(i) for i in ll.split()]
    ih,ia,ie,ii,modelin=rl[0:5]
    #print(rl)
    for j in range(1,nsour+1):
        alpha[ih,ia,ie,ii,j]=rl[4+j]
    model[ih,ia,ie,ii]=modelin

print("processed the file")

processed the file


In [ ]:
asteroid_elements=[]

nneo_float=6000000 # -1 to have the program calculate the "real" estimted numbers of asteroids
diam1=0.01         # minimum asteroid diameter
diam2=10.0         # maximum asteroid diameter

#Orbital bin sizes
da=(maxa-mina)/na
de=(maxe-mine)/ne
di=(maxi-mini)/ni

#construct the size distribution
nlog0=np.log10(nref)
nlogp[6]=nlog0+(dpoint[6]-ldref)*gamma[5] # dref falls into 5nd segment
nlogp[5]=nlog0+(dpoint[5]-ldref)*gamma[5]
nlogp[4]=nlogp[5]+(dpoint[4]-dpoint[5])*gamma[4]
nlogp[3]=nlogp[4]+(dpoint[3]-dpoint[4])*gamma[3]
nlogp[2]=nlogp[3]+(dpoint[2]-dpoint[3])*gamma[2]
nlogp[1]=nlogp[2]+(dpoint[1]-dpoint[2])*gamma[1]
for j in range(6,npoint+1):
    nlogp[j]=nlogp[j-1]+(dpoint[j]-dpoint[j-1])*gamma[j-1]

yp1=1.e30
ypn=1.e30

y2= spline(dpoint,nlogp,npoint,yp1,ypn)

#     Rescale distribution to the desired number of model NEOs (nneo)
ld1=np.log10(diam1)
ld2=np.log10(diam2)
ndiam1=splint(dpoint,nlogp,y2,npoint,ld1)
ndiam2=splint(dpoint,nlogp,y2,npoint,ld2)
nreal=10**ndiam1-10**ndiam2
if (nneo_float>=0):
      nneo=nneo_float
else:
    nneo=nreal

print("number of neos in range",diam1*1000,"to",diam2*1000,"meters: ",nneo)

#     determine maxmod (differential!)
#      dd=0.0001d0
#      ldleft=ld1
#      ldright=log10(diam1+dd)
dd=0.01
ldleft=ld1
ldright=ld1+dd
ndiam1=splint(dpoint,nlogp,y2,npoint,ldleft)
ndiam2=splint(dpoint,nlogp,y2,npoint,ldright)
maxmod=10.**ndiam1-10.**ndiam2

#     Generate NEOs
counter=0
while True:
    # Generate a logarithmic diameter value
    ldiam = ld1 + (ld2 - ld1) * random.random()
    ldleft = ldiam - 0.5 * dd
    ldright = ldiam + 0.5 * dd

    # Perform spline interpolation
    ndiam1 = splint(dpoint, nlogp, y2, npoint, ldleft)
    ndiam2 = splint(dpoint, nlogp, y2, npoint, ldright)

    # Compute differential weight
    wmod = 10.0**ndiam1 - 10.0**ndiam2

    # Rejection condition: If random value is greater than weight, try again
    if random.random() * maxmod > wmod:
        continue  # Retry (equivalent to `goto 300` in Fortran)

    # Generate albedo & magnitude

    # Albedo distribution from Wright+16
    # facd = 0.253
    # sig1 = 0.030
    # sig2 = 0.168

    # Debiased albedo distribution from Model319 (unchanging with D)
    # facd = 0.234
    # sig1 = 0.0287
    # sig2 = 0.170

    # Debiased albedo distribution from Model330 (1-3 km)
    # facd = 0.306
    # sig1 = 0.0251
    # sig2 = 0.162

    # Size-dependent albedo model
    logd = ldiam
    logd1 = np.log10(np.sqrt(0.1 * 0.3))
    logd2 = np.log10(np.sqrt(0.3 * 1.0))
    logd3 = np.log10(np.sqrt(1.0 * 3.0))

    # Model332
    if logd < logd1:
        facd = 0.183
        sig1 = 0.0566
        sig2 = 0.182

    # Models 332 -> 331
    if logd1 <= logd < logd2:
        facd = 0.183 + (logd - logd1) * (0.212 - 0.183) / (logd2 - logd1)
        sig1 = 0.0566 + (logd - logd1) * (0.0367 - 0.0566) / (logd2 - logd1)
        sig2 = 0.182 + (logd - logd1) * (0.182 - 0.182) / (logd2 - logd1)

    # Models 331 -> 330
    if logd2 <= logd < logd3:
        facd = 0.212 + (logd - logd2) * (0.306 - 0.212) / (logd3 - logd2)
        sig1 = 0.0367 + (logd - logd2) * (0.0251 - 0.0367) / (logd3 - logd2)
        sig2 = 0.182 + (logd - logd2) * (0.162 - 0.182) / (logd3 - logd2)

    # Model330
    if logd >= logd3:
        facd = 0.306
        sig1 = 0.0251
        sig2 = 0.162

    # Rejection method to select albedo
    while True:  # goto 400 logic
        pV = random.random()  # Random albedo between 0 and 1
        phi = facd * (pV * np.exp(-pV**2 / (2.0 * sig1 * sig1)) / (sig1 * sig1))
        phi += (1.0 - facd) * (pV * np.exp(-pV**2 / (2.0 * sig2 * sig2)) / (sig2 * sig2))

        if random.random() * 10.0 > phi:
            continue  # Reject this albedo value and retry
        break  # Accept this albedo value

    hmag = -5.0 * np.log10(10.0**ldiam * np.sqrt(pV) / 1329.0)

    # print(f"{10.0**ldiam} {pV} {hmag}")  # Equivalent to Fortran's commented write statement

    # Now get the orbit for hmag
    if minh <= hmag < maxh:
        hmag_fake = hmag  # Used for orbital distribution, no info outside model domain

    if hmag < minh:
        hmag_fake = minh + 1e-10

    if hmag > maxh:
        hmag_fake = maxh - 1e-10

    ih = int((hmag_fake - minh) / dh) + 1  # Last available bin for extrapolation

    # Determine maxmod2 for magnitude bin ih
    maxmod2 = 0.0
    for ia in range(1, na + 1):
        for ie in range(1, ne + 1):
            for ii in range(1, ni + 1):
                if model[ih, ia, ie, ii] > maxmod2:
                    maxmod2 = model[ih, ia, ie, ii]
                if ih > 1:
                    if model[ih - 1, ia, ie, ii] > maxmod2:
                        maxmod2 = model[ih - 1, ia, ie, ii]
                if ih < nh:
                    if model[ih + 1, ia, ie, ii] > maxmod2:
                        maxmod2 = model[ih + 1, ia, ie, ii]

    while True:   # goto 500 logic

        a=mina+random.random()*(maxa-mina)
        e=mine+random.random()*(maxe-mine)
        inc=mini+random.random()*(maxi-mini)
        if (a*(1-e)>1.3):
            continue # only NEOs
        ia=int((a-mina)/da)+1
        ie=int((e-mine)/de)+1
        ii=int((inc-mini)/di)+1

        # Power law (linear in log) interpolation for absolute magnitudes
        bh = minh + (ih - 0.5) * dh

        if hmag_fake > bh:
            ih0 = ih
            ih1 = ih + 1
        else:
            ih0 = ih - 1
            ih1 = ih

        v0 = minh + (ih0 - 0.5) * dh
        v1 = minh + (ih1 - 0.5) * dh

        # Border effects
        if ih0 < 1:
            ih0 = 1
        if ih1 > nh:
            ih1 = nh

        # Trilinear interpolation for orbital elements
        ba = mina + (ia - 0.5) * da
        be = mine + (ie - 0.5) * de
        bi = mini + (ii - 0.5) * di

        if a > ba:
            ia0 = ia
            ia1 = ia + 1
        else:
            ia0 = ia - 1
            ia1 = ia

        if e > be:
            ie0 = ie
            ie1 = ie + 1
        else:
            ie0 = ie - 1
            ie1 = ie

        if inc > bi:
            ii0 = ii
            ii1 = ii + 1
        else:
            ii0 = ii - 1
            ii1 = ii


        x0 = mina + (ia0 - 0.5) * da
        x1 = mina + (ia1 - 0.5) * da
        y0 = mine + (ie0 - 0.5) * de
        y1 = mine + (ie1 - 0.5) * de
        z0 = mini + (ii0 - 0.5) * di
        z1 = mini + (ii1 - 0.5) * di

        xd = (a - x0) / (x1 - x0)
        yd = (e - y0) / (y1 - y0)
        zd = (inc - z0) / (z1 - z0)

        if ia0 < 1:
            ia0 = 1
        if ia1 > na:
            ia1 = na
        if ie0 < 1:
            ie0 = 1
        if ie1 > ne:
            ie1 = ne
        if ii0 < 1:
            ii0 = 1
        if ii1 > ni:
            ii1 = ni


        # Step 1: Trilinear interpolation for ih0
        c000 = model[ih0, ia0, ie0, ii0]
        c100 = model[ih0, ia1, ie0, ii0]
        c010 = model[ih0, ia0, ie1, ii0]
        c001 = model[ih0, ia0, ie0, ii1]
        c111 = model[ih0, ia1, ie1, ii1]
        c011 = model[ih0, ia0, ie1, ii1]
        c101 = model[ih0, ia1, ie0, ii1]
        c110 = model[ih0, ia1, ie1, ii0]

        c00 = c000 * (1.0 - xd) + c100 * xd
        c01 = c001 * (1.0 - xd) + c101 * xd
        c10 = c010 * (1.0 - xd) + c110 * xd
        c11 = c011 * (1.0 - xd) + c111 * xd

        c0 = c00 * (1.0 - yd) + c10 * yd
        c1 = c01 * (1.0 - yd) + c11 * yd

        wmod0 = c0 * (1.0 - zd) + c1 * zd

        # Step 2: Trilinear interpolation for ih1
        c000 = model[ih1, ia0, ie0, ii0]
        c100 = model[ih1, ia1, ie0, ii0]
        c010 = model[ih1, ia0, ie1, ii0]
        c001 = model[ih1, ia0, ie0, ii1]
        c111 = model[ih1, ia1, ie1, ii1]
        c011 = model[ih1, ia0, ie1, ii1]
        c101 = model[ih1, ia1, ie0, ii1]
        c110 = model[ih1, ia1, ie1, ii0]

        c00 = c000 * (1.0 - xd) + c100 * xd
        c01 = c001 * (1.0 - xd) + c101 * xd
        c10 = c010 * (1.0 - xd) + c110 * xd
        c11 = c011 * (1.0 - xd) + c111 * xd

        c0 = c00 * (1.0 - yd) + c10 * yd
        c1 = c01 * (1.0 - yd) + c11 * yd

        wmod1 = c0 * (1.0 - zd) + c1 * zd

        #     Step 3: power law for magnitudes
        vd=(hmag_fake-v0)/(v1-v0)
        wmod=np.log10(wmod0+1.e-30)*(1.-vd)+np.log10(wmod1+1.e-30)*vd
        wmod=10**wmod

        if(wmod>maxmod2):
            print('Error: wmod>maxmod2, exiting...')
            exit()

        if(random.random()*maxmod2>wmod):
            continue #reject this orbit

        break

    asteroid_elements.append([a,e,inc,ldiam,pV,hmag])
    # print('{:5.2f} {:7.5f} {:5.3f} {:5.2f} {:7.4f} {:5.3f}'.format(hmag,a,e,inc,10.**ldiam,pV,a*(1-e),a*(1+e)))
    counter=counter+1
    if(counter>=nneo):
        break     # cycle until we generate the desired number of NEOs
    continue

number of neos in range 10.0 to 10000.0 meters:  6000000


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-34-649965f8bd01>", line None, in <cell line: 0>
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'KeyboardInterrupt' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/ultratb.py", line 1101, in get_records
    return _fixed_getinnerframes(etb, number_of_lines_of_context, tb_offset)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [34]:
i=0
for asteroid in asteroid_elements:
    if asteroid[0]*(1-asteroid[1])>1 or asteroid[0]*(1+asteroid[1])<1:
        continue
    i=i+1
    # print(i,asteroid)

In [ ]:
a1 = 1.00000011
count = 0;
e1 = 0.01671022
deg = np.pi/180
i1 = 0.00005*deg
Omega1 = -11.26064*deg
omega1 = 102.94719*deg
theta01 = 300*deg
oe1=[a1,e1,i1,Omega1,omega1,theta01]
moidList=[]; a2List=[]; e2List=[]; i2List=[]; Omega2List=[]; omega2List=[]; theta02List=[]; thetaf1List=[]; thetaf2List=[]; diamList=[];

with open("Asteroids_ID.txt", "w") as file:
    file.write("# MOID(LD) a2(AU) e2 i2(rad) Omega2(rad) omega2(rad) theta02(rad) Diameter(m)\n")

    for asteroid in asteroid_elements:

        a2 = asteroid[0]
        e2 = asteroid[1]
        i2 = asteroid[2]*deg
        Omega2 = random.random()*2*np.pi
        omega2 = random.random()*2*np.pi
        M0 = random.random()*2*np.pi
        theta02 = true_anomaly(M0,a2,e2)
        oe2=[a2,e2,i2,Omega2,omega2,theta02]
        moid_value = moid(oe1,oe2)
        if (moid_value*1500/4<1):
            count += 1
            a2List.append(a2)
            e2List.append(e2)
            i2List.append(i2)
            Omega2List.append(Omega2)
            omega2List.append(omega2)
            theta02List.append(theta02)
            moidList.append(moid_value)
            diamList.append(10.**asteroid[3])
            file.write(f"{count} {moid_value * 1500 / 4:.10f} {a2:.10f} {e2:.10f} {i2:.10f} {Omega2:.10f} {omega2:.10f} {theta02:.10f} {1000 * 10.**asteroid[3]:.2f}\n")


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-35-1ee50a8e2cf8>", line 25, in <cell line: 0>
    moid_value = moid(oe1,oe2)
                 ^^^^^^^^^^^^^
  File "<ipython-input-31-af766141f0de>", line None, in moid
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'KeyboardInterrupt' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/ultratb.py", line 1101, in get_records
    return _fixed_get

In [35]:
moidList=np.array(moidList); timesin=[]; timemin=[]; timesout=[]; distmin=[]; index=[]
nval=sum(moidList*1500/4<1)
t=np.linspace(0,10*365.25*24*3600,10000)
with open("Asteroids_App.txt", "w") as file:
    file.write("# DDi MMi YYYYi disti(LD) DDc MMc YYYYc distc(LD) velc(km/s) Diameter(m) DDo MMo YYYYo disto(LD)\n")

    for n in range(nval):
        P1, lvec1, t01, R1 = static_elements(a1*AU_TO_M,e1,i1,Omega1,omega1,theta01,MU)
        P2, lvec2, t02, R2 = static_elements(a2List[n]*AU_TO_M,e2List[n],i2List[n],Omega2List[n],omega2List[n],theta02List[n],MU)
        timesin_n, timemin_n, timesout_n, distmin_n = time_evolution(a1*AU_TO_M,e1,a2List[n]*AU_TO_M,e2List[n],P1,P2,t01,t02,R1,R2,lvec1,lvec2,t,delta,MU,n,diamList,file)


KeyboardInterrupt: 

In [ ]:
moidList=np.array(moidList)

In [ ]:
sum(moidList*1500/4<1)*60

In [ ]:
np.min(moidList*1500/4)